# Course 7: Geometric Deep Learning and Graphs
## Session 6: Group-equivariant networks

**Goal:** build a CNN that is equivariant to 90-degree rotations and compare it with a plain CNN on rotated digits.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) on small built-in or synthetic data. Cells labelled NOT RUN need a library, a dataset download or a GPU; run them on Kaggle or Colab.

### 1. Data: upright digits to train, rotated digits to test
The small 8x8 digits shipped with scikit-learn stand in for MNIST.

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

d = load_digits()
X = torch.tensor(d.images / 16.0, dtype=torch.float32)[:, None]      # (N, 1, 8, 8)
y = torch.tensor(d.target)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
Xte_rot = torch.cat([torch.rot90(Xte, k, (2, 3)) for k in range(4)]); yte_rot = yte.repeat(4)
print("train", Xtr.shape, "| rotated test", Xte_rot.shape)

### 2. A C4 lifting convolution
Apply the same filter in its four rotations. Rotating the image then rotates each feature map and cycles the four channels, and a maximum over the four removes the cycling.

In [ ]:
class C4Conv(nn.Module):
    """Lifting convolution followed by a max over the four rotations."""
    def __init__(self, c_in, c_out, k=3):
        super().__init__()
        self.w = nn.Parameter(torch.randn(c_out, c_in, k, k) * (2 / (c_in * k * k)) ** 0.5)
        self.b = nn.Parameter(torch.zeros(c_out))
    def forward(self, x):
        maps = [F.conv2d(x, torch.rot90(self.w, r, (2, 3)), self.b, padding=1) for r in range(4)]
        return torch.stack(maps, 0).max(0).values

class Net(nn.Module):
    def __init__(self, equivariant, c=24):
        super().__init__()
        conv = (lambda i, o: C4Conv(i, o)) if equivariant else (lambda i, o: nn.Conv2d(i, o, 3, padding=1))
        self.c1, self.c2, self.fc = conv(1, c), conv(c, c), nn.Linear(c, 10)
    def features(self, x):
        return F.relu(self.c2(F.relu(self.c1(x))))
    def forward(self, x):
        return self.fc(self.features(x).amax((2, 3)))            # global max pool: invariant to where and which way

torch.manual_seed(0)
net = Net(True); x = Xtr[:4]
f, f_rot = net.features(x), net.features(torch.rot90(x, 1, (2, 3)))
print("equivariance error:", (torch.rot90(f, 1, (2, 3)) - f_rot).abs().max().item())
assert torch.allclose(torch.rot90(f, 1, (2, 3)), f_rot, atol=1e-5)

### 3. Train both on upright digits only

In [ ]:
def train(model, epochs=30):
    opt = torch.optim.Adam(model.parameters(), lr=3e-3)
    for _ in range(epochs):
        for idx in torch.randperm(len(Xtr)).split(64):
            loss = F.cross_entropy(model(Xtr[idx]), ytr[idx]); opt.zero_grad(); loss.backward(); opt.step()
    acc = lambda A, b: (model(A).argmax(1) == b).float().mean().item()
    return acc(Xte, yte), acc(Xte_rot, yte_rot)

torch.manual_seed(0); plain = train(Net(False))
torch.manual_seed(0); equi = train(Net(True))
print("plain CNN : upright %.3f  rotated %.3f" % plain)
print("C4 CNN    : upright %.3f  rotated %.3f" % equi)
assert equi[1] > plain[1] + 0.2

Note: a few digits are ambiguous under rotation (6 and 9), so even a perfectly invariant model cannot reach 100% on the rotated set. That is a property of the task, not a bug.

### 4. The same with escnn, on rotated MNIST (NOT RUN here)

In [ ]:
# NOT RUN: needs `pip install escnn` and the rotated MNIST download.
from escnn import gspaces, nn as enn
r2 = gspaces.rot2dOnR2(N=8)                                   # rotations by multiples of 45 degrees
f_in = enn.FieldType(r2, [r2.trivial_repr])                   # a grey image is a scalar field
f_hid = enn.FieldType(r2, 16 * [r2.regular_repr])
f_out = enn.FieldType(r2, 32 * [r2.regular_repr])
model = enn.SequentialModule(
    enn.R2Conv(f_in, f_hid, kernel_size=5, padding=2), enn.ReLU(f_hid),
    enn.PointwiseMaxPool(f_hid, 2),
    enn.R2Conv(f_hid, f_out, kernel_size=5, padding=2), enn.ReLU(f_out),
    enn.GroupPooling(f_out),                                  # max over the group gives invariant features
)
# x = enn.GeometricTensor(images, f_in); features = model(x).tensor.amax((2, 3)); then a linear classifier.

### TODO
1. Train the plain CNN with rotation augmentation. How close does it get to the C4 CNN, and at what cost in epochs?
2. Count parameters in both models. Is the comparison fair?
3. Run the escnn model on rotated MNIST with N = 4 and N = 8 and compare.